## 4.3 Create a Training Client

In [1]:
import tinker
from tinker import types
from pydantic.v1.datetime_parse import parse_date as parse_date

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(
    base_model="meta-llama/Llama-3.2-1B",
    rank=32,
)
tokenizer = training_client.get_tokenizer()

c:\Users\afsha\miniconda3\envs\agentic_env\Lib\site-packages\tinker\_compat.py:48: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.datetime_parse import parse_date as parse_date
Your Tinker SDK version is outdated. Please upgrade to the latest version.


This creates a LoRA adapter (rank 32) attached to the frozen Llama-3.2-1B weights on Tinker's servers. The `training_client` object is your handle to that remote model. The tokenizer runs locally on your machine.

## 4.4 Define the Training Data

Just 7 handwritten examples:

In [ ]:
examples = [
    {"input": "banana split",      "output": "anana-bay plit-say"},
    {"input": "quantum physics",   "output": "uantum-qay ysics-phay"},
    {"input": "donut shop",        "output": "onut-day op-shay"},
    {"input": "pickle jar",        "output": "ickle-pay ar-jay"},
    {"input": "space exploration", "output": "ace-spay exploration-way"},
    {"input": "rubber duck",       "output": "ubber-ray uck-day"},
    {"input": "coding wizard",     "output": "oding-cay izard-way"},
]

In a real fine-tuning job you would use thousands or millions of examples. Here we use 7 to keep the demo simple and fast.

## 4.5 Tokenize with Weight Masks

Each example must be converted into a `Datum` — Tinker's format for a training example. The critical step is setting the **weight mask** so the model only learns to predict the Pig Latin output, not the English prompt.

In [ ]:
def process_example(example, tokenizer):
    prompt = f"English: {example['input']}\nPig Latin:"

    # Tokenize prompt — the model sees this but is NOT trained on it
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=True)
    prompt_weights = [0] * len(prompt_tokens)

    # Tokenize completion — the model IS trained to produce this
    completion_tokens = tokenizer.encode(
        f" {example['output']}\n\n", add_special_tokens=False
    )
    completion_weights = [1] * len(completion_tokens)

    # Concatenate and shift for next-token prediction
    tokens = prompt_tokens + completion_tokens
    weights = prompt_weights + completion_weights
    input_tokens = tokens[:-1]
    target_tokens = tokens[1:]
    weights = weights[1:]

    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens=input_tokens),
        loss_fn_inputs=dict(weights=weights, target_tokens=target_tokens),
    )

processed = [process_example(ex, tokenizer) for ex in examples]

**What is happening here:**

1. The prompt `"English: banana split\nPig Latin:"` is tokenized into tokens. Each gets weight `0`.
2. The completion `" anana-bay plit-say\n\n"` is tokenized. Each gets weight `1`. *(Note the leading space — it separates the completion from the colon.)*
3. The full token sequence is shifted by one position to create input/target pairs for next-token prediction.
4. The result is packaged as a `Datum` with `model_input` (what the model sees) and `loss_fn_inputs` (targets and weights for the loss function).

If we visualize the tokens around the transition from prompt to completion:

```
Input Token      Target Token     Weight
-------------------------------------------
' Latin'         ':'              0        ← still prompt
':'              ' an'            1        ← completion starts here
' an'            'ana'            1
'ana'            '-'              1
'-'              'bay'            1
'bay'            ' p'             1
...
```

The model learns: *"after seeing `English: banana split\nPig Latin:`, the next tokens should be `anana-bay plit-say`."*

## 4.6 Train

The training loop is remarkably simple — 6 steps, all 7 examples in each batch:

In [ ]:
import numpy as np

for step in range(6):
    # Forward + backward: compute gradients on Tinker's GPUs
    fwdbwd_future = training_client.forward_backward(
        processed, "cross_entropy"
    )
    # Optimizer step: update the LoRA adapter weights
    optim_future = training_client.optim_step(
        types.AdamParams(learning_rate=1e-4)
    )
    # Wait for results and compute loss
    fwdbwd_result = fwdbwd_future.result()
    optim_result = optim_future.result()

    logprobs = np.concatenate(
        [out['logprobs'].tolist()
         for out in fwdbwd_result.loss_fn_outputs]
    )
    weights = np.concatenate(
        [ex.loss_fn_inputs['weights'].tolist()
         for ex in processed]
    )
    loss = -np.dot(logprobs, weights) / weights.sum()
    print(f"Step {step}: loss = {loss:.4f}")

Step 0: loss = 6.4281
Step 1: loss = 5.7310
Step 2: loss = 4.6245
Step 3: loss = 3.4666
Step 4: loss = 2.4927
Step 5: loss = 1.9450


**What is happening on each step:**

1. `forward_backward` sends all 7 tokenized examples to Tinker. The server runs the forward pass through Llama-3.2-1B + the LoRA adapter, computes the cross-entropy loss (weighted by the weight mask), and backpropagates to get gradients for the adapter parameters. It returns a **future** — the computation happens asynchronously.
2. `optim_step` tells the server to update the adapter weights using Adam with learning rate `1e-4`. Also returns a future.
3. We call `.result()` on both futures to wait for completion.
4. We compute the weighted average loss from the returned log-probabilities. This should decrease over the 6 steps.

> **Note:** both API calls return futures immediately. We submit both before waiting, which allows them to pipeline on the server.

**Expected output (approximate):**
```
Step 0: loss = 8.2341
Step 1: loss = 5.1892
Step 2: loss = 3.0156
Step 3: loss = 1.8743
Step 4: loss = 1.1205
Step 5: loss = 0.7831
```

The loss drops sharply because we have only 7 examples and the model can memorize them quickly. In a real task with thousands of examples, the loss would decrease more gradually.

## 4.7 Sample from the Fine-Tuned Model

To generate text, we first save the current adapter weights and create a sampling client:

In [ ]:
sampler = training_client.save_weights_and_get_sampling_client(
    name="pig-latin-model"
)

prompt = types.ModelInput.from_ints(
    tokenizer.encode("English: coffee break\nPig Latin:")
)
params = types.SamplingParams(
    max_tokens=20, temperature=0.0, stop=["\n"]
)

result = sampler.sample(
    prompt=prompt, sampling_params=params, num_samples=8
).result()

print("Responses:")
for i, seq in enumerate(result.sequences):
    print(f"  {i}: {tokenizer.decode(seq.tokens)}")

Responses:
  0:  oof-bey  breakay


  1:  oof-bey  breakay


  2:  oof-bey  breakay


  3:  oof-bey  breakay


  4:  oof-bey  breakay


  5:  oof-bey  breakay


  6:  oof-bey  breakay


  7:  oof-bey  breakay




**Expected output (approximate):**
```
Responses:
  0:  offee-cay eak-bray
  1:  offey-cay eak-bray
  2:  offecay eakbray
  ...
```

The model has learned the Pig Latin pattern — it moves consonants to the end and appends "ay" — even for the phrase "coffee break" which was **not** in the training data. The outputs are not perfectly consistent (note the slight variations across samples), which reflects the fact that we trained on only 7 examples for 6 steps. More data and more training would improve consistency.

## 4.8 What Just Happened

Let's recap what the model actually learned:

- The original **Llama-3.2-1B weights are completely frozen**. Not a single one of its 1.2 billion parameters changed.
- A **LoRA adapter with rank 32** was attached to the model's weight matrices. Only these adapter parameters were trained — roughly 20 million parameters, or about 1.6% of the base model.
- After 6 training steps on 7 examples, the adapter learned a pattern: *"when you see `English: [X]\nPig Latin:`, produce the Pig Latin translation of X."*
- The adapter weights are a small file that could be saved, shared, and swapped without reloading the base model.

All of the actual computation — the forward passes, the gradient calculations, the weight updates — happened on **Tinker's GPU cluster**. The Python script on your laptop only sent data and received results over the network.